For the EEG Experiment the design matrix needs to:
- contain a lot more neutral cues (400)
- use only 4 images, either (_00 or _01)
- have balanced L/R target representation per image


For the EEG Experiment we need to come up with a scheme for the triggers 
We will have a 2 x 3 x 2 x 4 design (Masking x Expectation x L/R x images) - this gives me 48 unique triggers 
The triggers with Brain products can vary from 1 x 250 


- images-left: 0-1-2-3 \\
- images-right: 4-5-6-7 \\
- mask: + 10-20 \\
- expectation: decimals, 100s, 200s (neutral, expected, unexpected)\\


11 -> image 1: L side, early-mask, neutral \
111-> image 1: L side, early-mask, expected \
211-> image 1: L side, early-mask, unexpected \
15 -> image 1: R side, early-mask, neutral \
115-> image 1: R side, early-mask, expected \
215-> image 1: R side, early-mask, unexpected 


21 -> image 1: L side, late-mask, neutral \
121-> image 1: L side, late-mask, expected \
221-> image 1: L side, late-mask, unexpected \
25 -> image 1: R side, late-mask, neutral \
125-> image 1: R side, late-mask, expected \
225-> image 1: R side, late-mask, unexpected 

In [1]:
import numpy as np
import os
import random
import pandas as pd 

def create_cue_dynam(highProb=0.7, lowProb=0.3, neutral=1.0, trials_per_cue=40):
    
    # double the amount for EEG 
    trial_per_neutral = 2 * trials_per_cue
    
    cue_data = {"cue_names": ["Sea animal",  "Water vessel",  "Neutral"],
                "cue_letters": ["SA", "WV", "X"],
                "cue_color": [[0.4, 0.6, 1.0], [-0.4, -0.2, 0.4], [-1, -1, -1]],
                "cue_highProb_cats": [["dolphin", "whale"], ["speedboat", "submarine"], 
                                    ["dolphin", "whale", "speedboat", "submarine"]],
                
                "cue_lowProb_cats": [["speedboat", "submarine"], ["dolphin", "whale"],
                                    ["dolphin", "whale", "speedboat", "submarine"]]}

    cue_data["cue_highProb"] = []
    cue_data["cue_lowProb"] = []
    
    for cue_id, cue in enumerate(cue_data["cue_names"]):
        if cue != "Neutral":
            cue_data["cue_highProb"].append([np.round(highProb / len(cue_data["cue_highProb_cats"][cue_id]), 2)] * len(cue_data["cue_highProb_cats"][cue_id]))
            cue_data["cue_lowProb"].append([np.round(lowProb / len(cue_data["cue_lowProb_cats"][cue_id]), 2)] * len(cue_data["cue_lowProb_cats"][cue_id]))
        
        else:
            cue_data["cue_highProb"].append([np.round(neutral / len(cue_data["cue_highProb_cats"][cue_id]), 3)] * len(cue_data["cue_highProb_cats"][cue_id]))
            cue_data["cue_lowProb"].append([np.round(neutral / len(cue_data["cue_lowProb_cats"][cue_id]), 3)] * len(cue_data["cue_highProb_cats"][cue_id]))
            
    cue_data["high_prob_trials"] = [
    np.array(x) * (trial_per_neutral if cue_data["cue_names"][idx] == "Neutral" else trials_per_cue)
    for idx, x in enumerate(cue_data["cue_highProb"])
]
    cue_data["low_prob_trials"] =  [
    np.array(x) * (trial_per_neutral if cue_data["cue_names"][idx] == "Neutral" else trials_per_cue)
    for idx, x in enumerate(cue_data["cue_lowProb"])
]
    
    return cue_data

def assign_trigger(row, late=0.100):
    
    # ---- MASK ----
    if row['mask_ISI'] == 0.017:
        mask_code = 10
    elif row['mask_ISI'] == late:
        mask_code = 20
    else:
        raise ValueError("Unknown mask type")
    
    # ---- IMAGE (side dependent) ----
    image_id = row['image_index']
    
    # Left images should be 1–4
    # Right images should be 5–8
    if row['target_loc'] == 'L':
        image_code = image_id
    elif row['target_loc'] == 'R':
        image_code = image_id + 4
    else:
        raise ValueError("Unknown location")
    
    base_code = mask_code + image_code
    
    # ---- EXPECTATION ----
    if row['expectation'] == 'neutral':
        exp_code = 0
    elif row['expectation'] == 'expected':
        exp_code = 100
    elif row['expectation'] == 'unexpected':
        exp_code = 200
    else:
        raise ValueError("Unknown expectation")
    
    return base_code + exp_code

def create_changes(list_length, prec):
    # Number of instances to set as True (2%)
    num_true = int(list_length * prec)

    # Create lists of False values
    catch = [False] * list_length

    # Randomly select indices for fixing and imaging channels
    catch_indices = random.sample(range(list_length), num_true)

    for idx in catch_indices:
        catch[idx] = True
    
    return np.array(catch)

def build_constrained_order(df, seed=None, max_unexpected_run=1):
    rng = np.random.default_rng(seed)

    remaining = df.copy()
    ordered_rows = []

    last_target = None
    unexpected_run = 0

    while len(remaining) > 0:

        # valid candidates mask
        valid_mask = np.ones(len(remaining), dtype=bool)

        # Rule 1 — no same target twice
        if last_target is not None:
            valid_mask &= (remaining["target"].values != last_target)

        # Rule 2 — max unexpected run
        if unexpected_run >= max_unexpected_run:
            valid_mask &= (remaining["expectation"].values != "unexpected")

        valid = remaining[valid_mask]

        # if dead end → restart whole sequence
        if len(valid) == 0:
            return build_constrained_order(df, seed=rng.integers(0,1e9))

        # pick random valid row
        choice_idx = rng.integers(len(valid))
        row = valid.iloc[choice_idx]

        ordered_rows.append(row)

        # update state
        last_target = row["target"]
        if row["expectation"] == "unexpected":
            unexpected_run += 1
        else:
            unexpected_run = 0

        # remove selected row
        remaining = remaining.drop(valid.index[choice_idx])

    return pd.DataFrame(ordered_rows).reset_index(drop=True)

def create_block_trials(stim_path, cue_data, random_seed, long_isi=0.1, pick_images="_01", identity_catch=0.1): 
    categories = os.listdir(stim_path)
    stimuli = []
    for cat in categories:
        cat_path = os.path.join(stim_path, cat)
        files = os.listdir(cat_path)
        stimuli.extend([f".\\stimuli\\{cat}\\{x}" for x in files])

    # Filter stims
    stimuli = np.array(stimuli)
    stimuli = np.array([x for x in stimuli if pick_images in x ])
    stimuli = stimuli[np.argsort(stimuli)]
    
    data = {"target_id": [],
            "distractor_id": [],
            "target": [],
            "distractor": [],
            "expectation": [],
            "mask_ISI": [],
            "cue": [],
            "cue_color": [],
            "cue_letter": [],
            "target_name": [],
            "target_cat": [],
            "target_loc": []}

    mask_type = [0.017, long_isi]
    distractors = stimuli[np.char.count(stimuli, "mask") > 0]
    distractor_ids = np.arange(len(stimuli))[np.isin(stimuli, distractors)]
    local_distractor_ids = np.arange(0, len(distractors))

    for cue_id, cue in enumerate(cue_data["cue_names"]):
        high_cats = np.array(cue_data["cue_highProb_cats"][cue_id])
        low_cats = np.array(cue_data["cue_lowProb_cats"][cue_id])
        
        if cue != "Neutral":
            for i, l_cat in enumerate(low_cats):
                l_cat_stim = stimuli[np.char.count(stimuli, l_cat) > 0]
                targets = l_cat_stim[np.char.count(l_cat_stim, "mask") == 0]
                target_ids = np.arange(len(stimuli))[np.isin(stimuli, targets)]
            

                trial_collectos = []
                for mask in mask_type:
                
                    h_trials = int(cue_data["low_prob_trials"][cue_id][i])
                
                    data["target_id"].extend(np.repeat(target_ids , h_trials))
                    data["target"].extend(np.repeat(targets, h_trials))
                    data["target_loc"].extend(["L"] * int(h_trials // 2))
                    data["target_loc"].extend(["R"] * int(h_trials // 2)) 
                    
                    target_names =[x.split("\\")[-1] for x in targets]
                    target_categories = [x.split("_")[0] for x in target_names]
                    
                    random_distractors = np.random.choice(local_distractor_ids, h_trials)
                    data["distractor_id"].extend([distractor_ids[x] for x in random_distractors])
                    data["distractor"].extend([distractors[x] for x in random_distractors])
                    data["target_name"].extend(np.repeat(target_names , h_trials))
                    data["target_cat"].extend(np.repeat(target_categories , h_trials))
                    
                    if cue != "Neutral":
                        data["expectation"].extend(["unexpected"] * h_trials)
                    else:
                        data["expectation"].extend(["neutral"]* h_trials)
                        
                    data["mask_ISI"].extend([mask] * h_trials)
                    data["cue"].extend([cue] * h_trials)
                    data["cue_color"].extend([cue_data["cue_color"][cue_id]]* h_trials)
                    data["cue_letter"].extend([cue_data["cue_letters"][cue_id]]* h_trials)
                    
        for i, h_cat in enumerate(high_cats):
            h_cat_stim = stimuli[np.char.count(stimuli, h_cat) > 0]
            targets = h_cat_stim[np.char.count(h_cat_stim, "mask") == 0]
            target_ids = np.arange(len(stimuli))[np.isin(stimuli, targets)]

            trial_collectos = []
            for mask in mask_type:
                h_trials = int(cue_data["high_prob_trials"][cue_id][i])
                
                data["target_id"].extend(np.repeat(target_ids , h_trials))
                data["target"].extend(np.repeat(targets , h_trials))
                data["target_loc"].extend(["L"] * int(h_trials // 2)) 
                data["target_loc"].extend(["R"] * int(h_trials // 2)) 
                
                target_names =[x.split("\\")[-1] for x in targets]
                target_categories = [x.split("_")[0] for x in target_names]
                
                random_distractors = np.random.choice(local_distractor_ids, h_trials)
                data["distractor_id"].extend([distractor_ids[x] for x in random_distractors])
                data["distractor"].extend([distractors[x] for x in random_distractors])
                data["target_name"].extend(np.repeat(target_names , h_trials))
                data["target_cat"].extend(np.repeat(target_categories , h_trials))
                
                
                if cue != "Neutral":
                    data["expectation"].extend(["expected"] * h_trials)
                else:
                    data["expectation"].extend(["neutral"]* h_trials)
                    
                data["mask_ISI"].extend([mask] * h_trials)
                data["cue"].extend([cue] * h_trials)
                data["cue_color"].extend([cue_data["cue_color"][cue_id]]* h_trials)
                data["cue_letter"].extend([cue_data["cue_letters"][cue_id]] * h_trials)
                

                
    #data["target_loc"] = [np.random.choice(["L", "R"], 1, p=[0.5, 0.5])[0] for _ in range(len(data["target"]))]
    data["distractor_loc"] = ["R" if x == "L" else "L" for x in data["target_loc"]]
    mapping = {0: 1,
           7: 2,
           5: 3,
           3: 4}


    df = pd.DataFrame(data)
    df['image_index'] = df['target_id'].map(mapping)
    df['trigger'] = df.apply(assign_trigger, axis=1)
    df["identity_catch"] = create_changes(len(df), identity_catch)
    df = build_constrained_order(df, seed=random_seed)
    
    return df, stimuli

In [2]:
cue_data = create_cue_dynam()
stim_path = "/projects/crunchie/boyanova/EEG_Things/Mask_ExpAtt_EEG/stimuli"
trials, stimss = create_block_trials(stim_path, cue_data, random_seed=19)

In [15]:
unique_combos = trials[['image_index', 'target_name', 'target_id']].drop_duplicates()

In [16]:
unique_combos

,image_index,target_name,target_id
0,1,dolphin_01.jpg,0
1,3,submarine_01.jpg,5
2,4,whale_01.jpg,7
5,2,speedboat_01.jpg,3


In [12]:
trials.head()

,target_id,distractor_id,target,distractor,expectation,mask_ISI,cue,cue_color,cue_letter,target_name,target_cat,target_loc,distractor_loc,image_index,trigger,identity_catch
0,0,6,.\stimuli\dolphin\dolphin_01.jpg,.\stimuli\whale\mask-whale_01.png,neutral,0.100,Neutral,"[-1, -1, -1]",X,dolphin_01.jpg,dolphin,L,R,1,21,False
1,5,1,.\stimuli\submarine\submarine_01.jpg,.\stimuli\dolphin\mask-dolphin_01.png,expected,0.017,Water vessel,"[-0.4, -0.2, 0.4]",WV,submarine_01.jpg,submarine,R,L,3,117,False
2,7,4,.\stimuli\whale\whale_01.jpg,.\stimuli\submarine\mask-submarine_01.png,unexpected,0.017,Water vessel,"[-0.4, -0.2, 0.4]",WV,whale_01.jpg,whale,R,L,4,218,False
3,5,4,.\stimuli\submarine\submarine_01.jpg,.\stimuli\submarine\mask-submarine_01.png,neutral,0.100,Neutral,"[-1, -1, -1]",X,submarine_01.jpg,submarine,L,R,3,23,False
4,7,4,.\stimuli\whale\whale_01.jpg,.\stimuli\submarine\mask-submarine_01.png,unexpected,0.100,Water vessel,"[-0.4, -0.2, 0.4]",WV,whale_01.jpg,whale,R,L,4,228,False


In [11]:
unique_combos

,image_index,target_name
0,1,dolphin_01.jpg
1,3,submarine_01.jpg
2,4,whale_01.jpg
5,2,speedboat_01.jpg


In [1]:
import os
from PIL import Image

root_dir = "/projects/crunchie/boyanova/EEG_Things/Mask_ExpAtt_EEG/target_stimuli"
target_size = (500, 500)

# Image extensions to process
valid_exts = (".jpg", ".jpeg", ".png", ".bmp", ".tiff", ".webp")

for root, dirs, files in os.walk(root_dir):
    for file in files:
        if file.lower().endswith(valid_exts):
            img_path = os.path.join(root, file)

            try:
                with Image.open(img_path) as img:
                    img = img.convert("RGB")  # safe for consistency
                    img_resized = img.resize(target_size, Image.LANCZOS)
                    img_resized.save(img_path)

            except Exception as e:
                print(f"Failed to process {img_path}: {e}")
